## Single Node Slice

This is an introductory notebook that would randomly pick a Fabric site and launch a single node slice (experiment) on this site. 

The cells in this notebook represent different stages of the slice reservation process. In future notebooks, these cells will be merged into a single large cell with some minor variances depending on the specific purposes of these notebooks. 

### Verification

In the next cell, the `fablib` API will verify the token stored in `/home/fabric/.fabric/id_tojen.json` and the slice/bastion keys in `/home/fabric/.ssh/`. 

Information about the specific Fabric project, user ID, and where to find the token and keys are stored in `/home/fabric/.fabric/fabric_rc`. 

In [16]:
from pathlib import Path
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure
fablib.create_ssh_tunnel_config(overwrite=False)

### Slice Deployment

First, set a unique prefix to your slice. 


In [17]:
# change the user name to your username
slicePrefix = "ngo-single-" 

- Get a random site.
- Append the site name to your slice prefix
- Clean up left over slices with the same name
- Set up a new slice:
    - Specify a single node with name `node` on this slice.
    - Use default hardware configuration
    - Use `default_ubuntu_22` as image.
- Submit slice request
    - The options of `submit` surpress the provision logs (very verbose) and wait until the slice is ready
    - Attempt to delete the submitted slice (just in case) if submission fails
- Update slice status

In [18]:
# Get random site and append to slice name prefix
siteName = fablib.get_random_site()
sliceName = slicePrefix + siteName

# Clean up left-over slices with the same names
existing = None
for s in fablib.get_slices():
    if s.get_name() == sliceName:
        print(f"Deleting existing slice: {sliceName}")
        s.delete()
        break

# Set up new slice
slice = fablib.new_slice(name=sliceName)

# Specify a single node requirement for your slice
node = slice.add_node(name="node",site=siteName,image="default_ubuntu_22",)
    
# Submit slice request
try:
    print(f"Submitting slice at {siteName}...")
    slice.submit(progress=False, wait=True)
except Exception as e:
    print(f"{siteName}: slice submission failed with error {e}")
    try:
        slice.delete()
    except Exception:
        pass

# Update slice status
slice.update()
print(f"Slice state: {slice.get_state()}")
if slice.get_state() == "Closing":
    print(f"Need to find new site")
else: 
    print(f"Slice stable: {slice.isStable()}")

User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
Submitting slice at CIEN...
Running post boot config threads ...
Post boot config node, Done! (0 sec)
Saving fablib data...  Done!
Slice state: StableOK
Slice stable: True


### Slice information

We can print out all relevant information for the node, including the full format of an SSH command to get to that node from this environment

In [19]:
# Print out status and SSH command for all nodes
node = slice.get_node(name="node")
print("----", node.get_name(), "----")
print("management ip:", node.get_management_ip())
print(node.get_ssh_command())

---- node ----
management ip: 66.46.187.124
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@66.46.187.124


We can send command over to the node via `fablib`

In [20]:
stdout, stderr = node.execute("cat /proc/cpuinfo | grep processor", quiet=True)
print(stdout)

processor	: 0
processor	: 1



In [21]:
stdout, stderr = node.execute("cat /proc/meminfo | grep Mem", quiet=True)
print(stdout)

MemTotal:        8128336 kB
MemFree:         7654428 kB
MemAvailable:    7685516 kB



In [22]:
stdout, stderr = node.execute("df -h", quiet=True)
print(stdout)

Filesystem      Size  Used Avail Use% Mounted on
tmpfs           794M  1.2M  793M   1% /run
/dev/vda1       9.6G  1.6G  8.0G  17% /
tmpfs           3.9G     0  3.9G   0% /dev/shm
tmpfs           5.0M     0  5.0M   0% /run/lock
/dev/vda15      105M  6.1M   99M   6% /boot/efi
tmpfs           794M  4.0K  794M   1% /run/user/1000



**The default single node has:**
- 2 CPUs
- 8GB of RAM
- ~10G of Storage

### Delete Slice

Once complete, make sure to come back and delete your slices, returning resources to the community. 

In [23]:
# Delete slice
slice.delete()